# non-diff-fn-wrap — worked example 1: Wrap the non-differentiable sign function

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `non-diff-fn-wrap`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

When wrapping an operation that has no defined gradient (like `torch.sign`), the wrapper must still unbox MiniTensor inputs and re-box the output — but the output must have `requires_grad=False` and `recipe=None`, regardless of whether its inputs were tracked. This is because the backward pass has nowhere to go through a non-differentiable op: it cannot compute a useful gradient, and attaching a Recipe would cause the reverse-pass traversal to attempt a gradient that doesn't exist.

## Worked solution

**Step 1 — Unbox inputs.**
For each argument, if it is a `MiniTensor` we extract `.array` to get the raw numpy/torch array. Non-MiniTensor arguments pass through unchanged. This gives us the raw values to pass to the underlying function.

**Step 2 — Call the underlying function.**
We call `torch.sign` on the raw arguments. This returns a raw array.

**Step 3 — Apply the three-gate AND — and note the `is_differentiable=False` gate.**
The three conditions are: (a) global grad tracking is on, (b) `is_differentiable` is True, and (c) at least one input has `requires_grad=True`. Because `is_differentiable=False` for `sign`, the entire AND short-circuits to `False`. So `requires_grad` is `False` no matter what.

**Step 4 — Box without a Recipe.**
We create `MiniTensor(out_arr, requires_grad=False)`. Because `requires_grad` is False, we skip the Recipe construction. `out.recipe` stays `None`.

**Step 5 — Verify.**
We check that `out.requires_grad is False` and `out.recipe is None` even when the input was a tracked tensor.

In [ ]:
import torch
import numpy as np

# ---- Minimal MiniTensor / Recipe scaffold ----
class Recipe:
    def __init__(self, func, args, kwargs, parents):
        self.func = func; self.args = args
        self.kwargs = kwargs; self.parents = parents

class MiniTensor:
    def __init__(self, array, requires_grad: bool = False):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = None

grad_tracking_enabled = True

# ---- Implementation ----
def wrap_forward_fn(fwd_fn, is_differentiable: bool = True):
    def tensor_func(*args, **kwargs):
        raw_args = tuple(a.array if isinstance(a, MiniTensor) else a for a in args)
        out_arr = fwd_fn(*raw_args, **kwargs)
        requires_grad = (
            grad_tracking_enabled
            and is_differentiable
            and any(isinstance(a, MiniTensor) and a.requires_grad for a in args)
        )
        out = MiniTensor(out_arr, requires_grad)
        if requires_grad:
            parents = {i: a for i, a in enumerate(args) if isinstance(a, MiniTensor)}
            out.recipe = Recipe(fwd_fn, raw_args, kwargs, parents)
        return out
    return tensor_func

# Wrap differentiable and non-differentiable ops
add_wrap  = wrap_forward_fn(torch.add)                   # differentiable
sign_wrap = wrap_forward_fn(torch.sign, is_differentiable=False)  # not differentiable

# Exercise: tracked input through sign
x = MiniTensor(torch.tensor([1.5, -0.3, 0.0]), requires_grad=True)

result_add  = add_wrap(x, x)   # should have requires_grad=True, recipe set
result_sign = sign_wrap(x)     # should have requires_grad=False, recipe=None

print(f'add  requires_grad: {result_add.requires_grad}')   # True
print(f'add  recipe set  : {result_add.recipe is not None}')  # True
print(f'sign requires_grad: {result_sign.requires_grad}')  # False
print(f'sign recipe set  : {result_sign.recipe is not None}')  # False
assert result_sign.requires_grad is False
assert result_sign.recipe is None